Importing libraries

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o device: {device}")

Usando o device: cpu


In [ ]:
class AmazonDimensionsDataset(Dataset):
    def __init__(self, csv_file='data/cubagem_40k_amazon.csv', split='train', transform=None, categorias = []):
        self.df = pd.read_csv(csv_file)

        if len(categorias):
            self.df = self.df[self.df['categories'].addisin(categorias)]


        if 'split' in self.df.columns:
            self.df = self.df[self.df['split'] == split]
        
        self.transform = transform
        self.df = self.df.dropna(subset=['image_url', 'length_cm', 'width_cm', 'height_cm'])
        self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_url = row['image_url']
        
        try:
            response = requests.get(img_url, timeout=5)
            img = Image.open(BytesIO(response.content)).convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % len(self))
            
        if self.transform:
            img = self.transform(img)
            
        # Ordenamos as dimensões (Maior, Média, Menor) para evitar confusão de rotação
        dims = sorted([
            row['length_cm'], 
            row['width_cm'], 
            row['height_cm']
        ], reverse=True)
        
        targets = torch.tensor(dims, dtype=torch.float32)
        return img, targets

In [22]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cat = ["Beauty & Personal Care > Skin Care > Body > Moisturizers > Lotions", "Pet Supplies > Dogs > Beds & Furniture > Beds"]

train_dataset = AmazonDimensionsDataset('data/cubagem_40k_amazon.csv', split='train', transform=transform, categorias = cat)
val_dataset = AmazonDimensionsDataset('data/cubagem_40k_amazon.csv', split='val', transform=transform, categorias = cat)

print(len(train_dataset))
print(len(val_dataset))


train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

# 4. Configuração do Modelo
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# PASSO 1: Congela TODOS os pesos da ResNet base
for param in model.parameters():
    param.requires_grad = False

# Pega o número de features que saem da camada convolucional (512 na resnet18)
num_ftrs = model.fc.in_features

# PASSO 2: Substitui a 'fc' por uma MLP (Multi-Layer Perceptron)
# Como criamos isso DEPOIS do loop acima, esses novos pesos terão requires_grad=True por padrão
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.3),          # Opcional: Dropout ajuda muito a evitar overfitting na MLP
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 3)         # Saída final para as 3 dimensões
)

model = model.to(device)

# 5. Loss Function e Optimizer
criterion = nn.SmoothL1Loss().to(device)

# PASSO 3: O pulo do gato! Passamos para o otimizador APENAS os parâmetros da nossa nova MLP (model.fc)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

125
25


In [23]:
# 6. Loop de Treinamento e Validação em Batches
# 6. Loop de Treinamento e Validação em Batches
num_epochs = 10

for epoch in range(num_epochs):
    # --- TREINAMENTO ---
    model.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for inputs, targets in train_bar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        train_bar.set_postfix({'loss': loss.item()})
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    
    # --- VALIDAÇÃO ---
    model.eval()
    running_val_loss = 0.0
    
    # Variáveis para guardar as predições de amostra
    sample_preds = None
    sample_targets = None
    
    val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(val_bar):
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_val_loss += loss.item() * inputs.size(0)
            val_bar.set_postfix({'loss': loss.item()})
            
            # Salva o primeiro batch para a visualização
            if i == 0:
                sample_preds = outputs.cpu().numpy()
                sample_targets = targets.cpu().numpy()
                
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    
    print(f'\nEpoch {epoch+1} Results | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')
    
    # --- COMPARAÇÃO VISUAL ---
    print("-" * 65)
    print(f"{'Previsto (Max, Med, Min) em cm':<32} | {'Real (Max, Med, Min) em cm'}")
    print("-" * 65)
    
    # Pegamos os 5 primeiros itens desse batch guardado
    for p, t in zip(sample_preds[:5], sample_targets[:5]):
        p_str = f"[{p[0]:.1f}, {p[1]:.1f}, {p[2]:.1f}]"
        t_str = f"[{t[0]:.1f}, {t[1]:.1f}, {t[2]:.1f}]"
        print(f"{p_str:<32} | {t_str}")
    print("\n")

Epoch 1/10 [Val]: 100%|██████████| 7/7 [00:04<00:00,  1.46it/s, loss=14.2]



Epoch 1 Results | Train Loss: 22.2420 | Val Loss: 14.2603
-----------------------------------------------------------------
Previsto (Max, Med, Min) em cm   | Real (Max, Med, Min) em cm
-----------------------------------------------------------------
[46.6, 37.9, 10.3]               | [61.0, 61.0, 20.3]
[27.5, 22.3, 5.9]                | [12.0, 11.0, 10.0]
[25.6, 20.8, 5.5]                | [39.6, 26.1, 4.0]
[26.7, 21.6, 5.7]                | [19.5, 19.5, 7.8]




Epoch 2/10 [Val]: 100%|██████████| 7/7 [00:03<00:00,  2.06it/s, loss=6.24]



Epoch 2 Results | Train Loss: 16.2154 | Val Loss: 10.8665
-----------------------------------------------------------------
Previsto (Max, Med, Min) em cm   | Real (Max, Med, Min) em cm
-----------------------------------------------------------------
[57.3, 37.9, 7.7]                | [61.0, 61.0, 20.3]
[21.8, 14.0, 2.7]                | [12.0, 11.0, 10.0]
[28.0, 18.3, 3.4]                | [39.6, 26.1, 4.0]
[14.8, 9.3, 1.8]                 | [19.5, 19.5, 7.8]




Epoch 3/10 [Val]: 100%|██████████| 7/7 [00:02<00:00,  2.34it/s, loss=9.93]


...


Epoch 8 Results | Train Loss: 12.1011 | Val Loss: 8.8458
-----------------------------------------------------------------
Previsto (Max, Med, Min) em cm   | Real (Max, Med, Min) em cm
-----------------------------------------------------------------
[70.3, 51.1, 13.3]               | [61.0, 61.0, 20.3]
[21.9, 13.1, 5.0]                | [12.0, 11.0, 10.0]
[36.1, 25.2, 6.0]                | [39.6, 26.1, 4.0]
[15.5, 7.8, 4.3]                 | [19.5, 19.5, 7.8]




Epoch 9/10 [Val]: 100%|██████████| 7/7 [00:03<00:00,  2.01it/s, loss=8.47]



Epoch 9 Results | Train Loss: 13.9893 | Val Loss: 9.5983
-----------------------------------------------------------------
Previsto (Max, Med, Min) em cm   | Real (Max, Med, Min) em cm
-----------------------------------------------------------------
[78.6, 56.0, 20.3]               | [61.0, 61.0, 20.3]
[25.4, 14.8, 7.7]                | [12.0, 11.0, 10.0]
[43.4, 29.9, 10.0]               | [39.6, 26.1, 4.0]
[17.8, 8.5, 6.5]                 | [19.5, 19.5, 7.8]




Epoch 10/10 [Val]: 100%|██████████| 7/7 [00:03<00:00,  2.14it/s, loss=2.81]


Epoch 10 Results | Train Loss: 12.7126 | Val Loss: 10.8512
-----------------------------------------------------------------
Previsto (Max, Med, Min) em cm   | Real (Max, Med, Min) em cm
-----------------------------------------------------------------
[45.0, 32.0, 13.4]               | [61.0, 61.0, 20.3]
[16.5, 9.2, 5.8]                 | [12.0, 11.0, 10.0]
[23.5, 15.5, 6.1]                | [39.6, 26.1, 4.0]
[11.5, 5.1, 5.0]                 | [19.5, 19.5, 7.8]


